In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path(r"e:\0projects\0000Testing-and-Maintenance\F-main\TM2026")
OMNI_CSV     = PROJECT_ROOT / r"omni\anomaly_results.csv"
DATASET      = PROJECT_ROOT / "data" / "datasets" / "online_boutique_rca_full_v1"

In [3]:
print("Project root:", PROJECT_ROOT)
print("Omni CSV path:", OMNI_CSV)
print("Dataset path:", DATASET)

Project root: e:\0projects\0000Testing-and-Maintenance\F-main\TM2026
Omni CSV path: e:\0projects\0000Testing-and-Maintenance\F-main\TM2026\omni\anomaly_results.csv
Dataset path: e:\0projects\0000Testing-and-Maintenance\F-main\TM2026\data\datasets\online_boutique_rca_full_v1


In [4]:
sys.path.insert(0, str(PROJECT_ROOT))
from benchmark.pipeline import Pipeline, DatasetBundle

# 1) 加载数据集（提供真值/incidents，评估和画图都靠它）
bundle = DatasetBundle.load(DATASET)

# 2) 读 omni 原始分数
omni = pd.read_csv(OMNI_CSV)[["timestamp", "anomaly_score", "y_true"]].copy()

# 3) 预处理-A：自动判方向。OmniAnomaly 的 score 是对数似然(越大越正常)，
#    本 benchmark 约定"越大越异常"。若异常点均值 < 正常点均值，则取负对齐。
mu_anom = omni.loc[omni.y_true == 1, "anomaly_score"].mean()
mu_norm = omni.loc[omni.y_true == 0, "anomaly_score"].mean()
flip = mu_anom < mu_norm
omni["score"] = -omni["anomaly_score"] if flip else omni["anomaly_score"]
print(f"异常均值={mu_anom:.1f} 正常均值={mu_norm:.1f} -> {'取负对齐' if flip else '方向已正确'}")

# 4) 预处理-B：时间戳对齐到数据集 test 集。
#    omni 因窗口预热砍掉了 test 开头若干点，用 UTC 时间做 key 对齐，
#    omni 没覆盖的点分数留空(pipeline 内部会填 0 -> 判正常 -> 这些 incident 如实漏检)。
key = lambda s: pd.to_datetime(s, utc=True)
sub = bundle.test_x[["timestamp"]].copy()
sub["_k"] = key(sub["timestamp"])
omni["_k"] = key(omni["timestamp"])
sub = sub.merge(omni[["_k", "score"]], on="_k", how="left")
submission = sub.rename(columns={"score": "anomaly_score"})[["timestamp", "anomaly_score"]]

covered = submission["anomaly_score"].notna().sum()
print(f"数据集 test 点数={len(submission)}，omni 覆盖={covered}，预热缺失={len(submission)-covered}")

# 5) 跑官方 pipeline（submission 模式：用现成分数，不训练）
pipe = Pipeline(
    bundle,
    run_name="omni",
    output_root=PROJECT_ROOT / "output",
    scale_features=False,       # 用现成分数，无需特征缩放
    threshold_mode="best_f1",   # 与 my_detector 同口径
)
result = pipe.run(submission=submission)
print("\n输出目录:", result.output_dir)

异常均值=-9335.8 正常均值=-9127.5 -> 取负对齐
数据集 test 点数=2867，omni 覆盖=2768，预热缺失=99

 Pipeline result -> e:\0projects\0000Testing-and-Maintenance\F-main\TM2026\output\20260610_133037_omni
 Point-wise F1     : 0.297   (P=0.178 R=0.899)
 AUPRC             : 0.198
 AUROC             : 0.545
 Event Recall      : 0.925   (37/40 incidents)
 Recall@30s        : 0.925
 False alarms/hour : 591.579
 Median delay (s)  : 1.000
 Missed incidents  : 3 -> ['online_boutique_rca_6/INC-001', 'online_boutique_rca_7/INC-003', 'online_boutique_rca_7/INC-017']
 Threshold         : 7844.9230 [best_f1, deployable=False]
 Artifacts         : metrics.json, predictions.csv, per_incident.csv,
                     score_timeline.png, roc_pr_curves.png, score_distribution.png

输出目录: e:\0projects\0000Testing-and-Maintenance\F-main\TM2026\output\20260610_133037_omni
